# Sign Language Feature Encoder Training

## Overview 🚀

This notebook trains a reusable hand-motion encoder for dynamic sign retrieval. It first learns general temporal representations from the Google Isolated Sign Language Recognition corpus, then adapts the encoder to an object-NPY landmark collection. The temporary classification heads provide training supervision; exported features are not tied to a fixed vocabulary.

**Input contract:** two hands × 21 landmarks × `(x, y, z, velocity)`, up to 64 frames.  
**Output contract:** 128-dimensional L2-normalized features for every valid frame and zeros for padding.

Run the notebook from top to bottom on a Kaggle **single T4** runtime. Production defaults follow the `hand168-temporal` preprocessing contract used by the C++ runtime; set `SMOKE_TEST = True` for a short integration run. Dataset paths can be set explicitly in the configuration cell or discovered under `/kaggle/input`.

Data sources:

- [Google - Isolated Sign Language Recognition](https://www.kaggle.com/competitions/asl-signs/data)
- [ASL-preprocessing 7 output](https://www.kaggle.com/code/abdelrhmankaram/asl-preprocessing-7/output)


## Configuration ⚙️

All user-editable settings live in this cell. Explicit paths are recommended for reproducible public notebooks.


In [ ]:
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, Iterator, List, Mapping, Optional, Sequence, Tuple
import contextlib, csv, gc, hashlib, io, json, math, os, pickle, random, shutil, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from tqdm.auto import tqdm

PREPROCESSING_CONTRACT = "hand168-temporal"
CHECKPOINT_SCHEMA = 1
SMOKE_TEST = False

@dataclass
class TrainSettings:
    epochs: int
    batches_per_epoch: int
    classes_per_batch: int
    samples_per_class: int
    learning_rate: float
    freeze_epochs: int = 0
    dtw_every: int = 0

@dataclass
class Config:
    source_root: Optional[str] = None
    target_landmarks: Optional[str] = None
    work_dir: str = "/kaggle/working/signlang-det"
    seed: int = 20260712
    max_frames: int = 64
    min_frames: int = 12
    max_input_frames: int = 120
    feature_dim: int = 168
    embedding_dim: int = 128
    padding_value: float = -100.0
    num_workers: int = 0
    cache_workers: int = 4
    cache_chunk_size: int = 256
    cache_flush_chunks: int = 4
    weight_decay: float = 1e-4
    use_amp: bool = True
    use_data_parallel: bool = False
    resume: bool = True
    cleanup_epoch_checkpoints: bool = True
    keep_latest_checkpoint: bool = False
    top_k: int = 20
    dtw_window: int = 12
    source_reference_per_class: int = 1
    source_query_per_class: int = 4
    final_query_per_class: int = 2
    target_right_slice: Tuple[int, int] = (0, 21)
    target_left_slice: Tuple[int, int] = (21, 42)
    source: TrainSettings = field(default_factory=lambda: TrainSettings(30, 300, 16, 4, 3e-4))
    adaptation: TrainSettings = field(default_factory=lambda: TrainSettings(40, 200, 16, 2, 1e-4, 2, 5))

CFG = Config()
if SMOKE_TEST:
    CFG.source = TrainSettings(1, 2, 2, 2, 3e-4)
    CFG.adaptation = TrainSettings(1, 2, 2, 2, 1e-4, 0, 1)

WORK_DIR = Path(CFG.work_dir)
WORK_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG.seed)
print(json.dumps(asdict(CFG), indent=2, default=str))


In [ ]:
# Number of deterministic target-training samples packaged for RKNN INT8 calibration.
int8_calibration_samples: int = 100
CFG.int8_calibration_samples = int8_calibration_samples


## Environment Check 🔍

CUDA availability alone is insufficient. The selected GPU architecture must also be present in the installed PyTorch build.


In [ ]:
GOOGLE_ASL_URL = "https://www.kaggle.com/competitions/asl-signs/data"
TARGET_OUTPUT_URL = "https://www.kaggle.com/code/abdelrhmankaram/asl-preprocessing-7/output"
STANDARD_SOURCE_RELATIVE = "competitions/asl-signs"
STANDARD_TARGET_RELATIVE = "notebooks/abdelrhmankaram/asl-preprocessing-7/landmarks"

def validate_source_root(path: Path) -> Path:
    path = path.expanduser().resolve()
    required = [path / "train.csv", path / "sign_to_prediction_index_map.json", path / "train_landmark_files"]
    missing = [str(item) for item in required if not item.exists()]
    if missing:
        raise FileNotFoundError(
            f"Google ASL dataset validation failed at {path}. Missing: {missing}. Source: {GOOGLE_ASL_URL}"
        )
    with (path / "train.csv").open("r", newline="", encoding="utf-8") as handle:
        columns = set(next(csv.reader(handle), []))
    expected = {"path", "sign", "participant_id"}
    if not expected.issubset(columns):
        raise ValueError(f"Google ASL train.csv is missing columns {sorted(expected - columns)} at {path}")
    return path

def validate_target_landmarks(path: Path) -> Path:
    path = path.expanduser().resolve()
    if not path.is_dir() or next(path.glob("*.npy"), None) is None:
        raise FileNotFoundError(
            f"ASL-preprocessing 7 validation failed at {path}; expected landmarks/*.npy. Source: {TARGET_OUTPUT_URL}"
        )
    return path

def _valid_source_candidates(input_root: Path) -> List[Path]:
    candidates = []
    for marker in input_root.rglob("sign_to_prediction_index_map.json"):
        try: candidates.append(validate_source_root(marker.parent))
        except (FileNotFoundError, ValueError): pass
    return sorted(set(candidates))

def _valid_target_candidates(input_root: Path) -> List[Path]:
    candidates = []
    for path in input_root.rglob("landmarks"):
        try: candidates.append(validate_target_landmarks(path))
        except FileNotFoundError: pass
    return sorted(set(candidates))

def _standard_or_error(input_root: Path, relative: str, validator, candidates, description: str, url: str) -> Path:
    standard = input_root / relative
    if standard.exists():
        return validator(standard)
    alternatives = candidates(input_root)
    preview = "\n".join(f"  - {path}" for path in alternatives[:20])
    if len(alternatives) > 1:
        raise FileNotFoundError(
            f"Multiple {description} candidates were found, but the standard Kaggle mount is missing: {standard}. "
            f"Set the corresponding Config path explicitly. Source: {url}\n{preview}"
        )
    if len(alternatives) == 1:
        raise FileNotFoundError(
            f"The standard Kaggle mount is missing: {standard}. A non-standard candidate was found at "
            f"{alternatives[0]}; set the corresponding Config path explicitly. Source: {url}"
        )
    raise FileNotFoundError(f"The standard Kaggle mount is missing: {standard}. Attach the data from {url}")

def resolve_inputs(cfg: Config, input_root: Path = Path("/kaggle/input")) -> Tuple[Path, Path]:
    if not input_root.is_dir():
        raise FileNotFoundError("/kaggle/input is unavailable. Attach both required Kaggle data sources.")
    source = validate_source_root(Path(cfg.source_root)) if cfg.source_root else _standard_or_error(
        input_root, STANDARD_SOURCE_RELATIVE, validate_source_root, _valid_source_candidates,
        "source dataset", GOOGLE_ASL_URL,
    )
    target = validate_target_landmarks(Path(cfg.target_landmarks)) if cfg.target_landmarks else _standard_or_error(
        input_root, STANDARD_TARGET_RELATIVE, validate_target_landmarks, _valid_target_candidates,
        "target landmark dataset", TARGET_OUTPUT_URL,
    )
    return source, target

def validate_cuda_build() -> torch.device:
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required. Select a Kaggle T4 accelerator and restart the session.")
    index = torch.cuda.current_device()
    capability = torch.cuda.get_device_capability(index)
    compiled = set(torch.cuda.get_arch_list())
    arch = f"sm_{capability[0]}{capability[1]}"
    if compiled and arch not in compiled:
        raise RuntimeError(
            f"{torch.cuda.get_device_name(index)} requires {arch}, but this PyTorch build contains {sorted(compiled)}. "
            "Use a T4-compatible Kaggle image."
        )
    print(f"GPU: {torch.cuda.get_device_name(index)} | capability: {arch} | PyTorch: {torch.__version__}")
    return torch.device("cuda", index)

DEVICE = validate_cuda_build()
SOURCE_ROOT, TARGET_LANDMARKS = resolve_inputs(CFG)
print(f"Source dataset: {SOURCE_ROOT}\nTarget landmarks: {TARGET_LANDMARKS}\nOutputs: {WORK_DIR}")


## Shared Preprocessing 🧩

`hand168-temporal` matches the `SegmentPreprocessor` contract used by the C++ runtime. Hand identity comes from source metadata, never horizontal image position.


In [ ]:
def hand_present(hand: np.ndarray) -> np.ndarray:
    return np.any(np.abs(hand) > 0, axis=(1, 2))

def trim_empty_edges(hands: np.ndarray) -> np.ndarray:
    present = hand_present(hands[:, 0]) | hand_present(hands[:, 1])
    indices = np.flatnonzero(present)
    return hands[indices[0]:indices[-1] + 1] if len(indices) else hands[:0]

def resample_normalized(coords: np.ndarray, present: np.ndarray, length: int = 64) -> Tuple[np.ndarray, np.ndarray]:
    if len(coords) == length:
        return coords.copy(), present.copy()
    output = np.zeros((length, 2, 21, 3), np.float32)
    output_present = np.zeros((length, 2), bool)
    for out_index in range(length):
        source = out_index * (len(coords) - 1) / (length - 1)
        lower = int(math.floor(source)); upper = min(lower + 1, len(coords) - 1)
        weight = source - lower
        nearest = min(int(math.floor(source + 0.5)), len(coords) - 1)
        for hand in range(2):
            output_present[out_index, hand] = present[nearest, hand]
            if not output_present[out_index, hand]:
                continue
            left = coords[lower, hand] if present[lower, hand] else coords[nearest, hand]
            right = coords[upper, hand] if present[upper, hand] else coords[nearest, hand]
            output[out_index, hand] = left + (right - left) * weight
    return output, output_present

def normalize_coordinates(hands: np.ndarray, eps: float = 1e-6) -> Tuple[np.ndarray, np.ndarray]:
    coords = np.zeros_like(hands, dtype=np.float32)
    present = np.zeros((len(hands), 2), dtype=bool)
    for h in range(2):
        present[:, h] = hand_present(hands[:, h])
        relative = hands[:, h] - hands[:, h, :1]
        scale = np.linalg.norm(relative, axis=-1).max(axis=1)
        coords[present[:, h], h] = relative[present[:, h]] / (scale[present[:, h], None, None] + eps)
    return coords, present

def preprocess_hands(
    hands: np.ndarray, max_frames: int = 64, min_frames: int = 12,
    padding_value: float = -100.0, max_input_frames: int = 120
) -> Tuple[np.ndarray, int]:
    hands = np.asarray(hands, dtype=np.float32)
    if hands.ndim != 4 or hands.shape[1:] != (2, 21, 3):
        raise ValueError(f"Expected T x 2 x 21 x 3 hands, received {hands.shape}")
    if not np.isfinite(hands).all():
        raise ValueError("Landmarks contain non-finite values")
    hands = trim_empty_edges(hands)
    if len(hands) < min_frames:
        raise ValueError(f"Sequence has {len(hands)} valid action frames; minimum is {min_frames}")
    if len(hands) > max_input_frames:
        raise ValueError(f"Sequence has {len(hands)} valid action frames; maximum is {max_input_frames}")
    coords, present = normalize_coordinates(hands)
    if len(hands) > max_frames:
        coords, present = resample_normalized(coords, present, max_frames)
    velocity = np.zeros_like(coords)
    continuous = present[1:] & present[:-1]
    delta = coords[1:] - coords[:-1]
    for h in range(2):
        velocity[1:, h][continuous[:, h]] = delta[:, h][continuous[:, h]]
    features = np.concatenate([coords, np.linalg.norm(velocity, axis=-1, keepdims=True)], axis=-1)
    valid_length = len(features)
    padded = np.full((max_frames, 2, 21, 4), padding_value, dtype=np.float32)
    padded[:valid_length] = features
    return padded.reshape(max_frames, 168), valid_length

def pack_hand_rows(frame_values, hand_types, landmark_indices, coordinates) -> np.ndarray:
    frame_values = np.asarray(frame_values)
    hand_types = np.asarray(hand_types)
    landmark_indices = np.asarray(landmark_indices, dtype=np.int64)
    coordinates = np.asarray(coordinates, dtype=np.float32)
    if not (len(frame_values) == len(hand_types) == len(landmark_indices) == len(coordinates)):
        raise ValueError("Parquet hand columns have inconsistent lengths")
    if not len(frame_values):
        return np.zeros((0, 2, 21, 3), np.float32)
    _, frame_indices = np.unique(frame_values, return_inverse=True)
    hand_indices = np.full(len(hand_types), -1, dtype=np.int8)
    hand_indices[hand_types == "left_hand"] = 0
    hand_indices[hand_types == "right_hand"] = 1
    valid = (
        (hand_indices >= 0) & (landmark_indices >= 0) & (landmark_indices < 21)
        & np.isfinite(coordinates).all(axis=1)
    )
    output = np.zeros((int(frame_indices.max()) + 1, 2, 21, 3), np.float32)
    output[frame_indices[valid], hand_indices[valid], landmark_indices[valid]] = coordinates[valid]
    return output

def source_parquet_hands(path: Path) -> np.ndarray:
    frame = pd.read_parquet(path, columns=["frame", "type", "landmark_index", "x", "y", "z"])
    frame = frame[frame["type"].isin(["left_hand", "right_hand"])].drop_duplicates(
        ["frame", "type", "landmark_index"], keep="last"
    )
    return pack_hand_rows(
        frame["frame"].to_numpy(), frame["type"].to_numpy(),
        frame["landmark_index"].to_numpy(), frame[["x", "y", "z"]].to_numpy(dtype=np.float32),
    )


## Source Dataset Preparation 💾

Data is loaded from [Google - Isolated Sign Language Recognition](https://www.kaggle.com/competitions/asl-signs/data). Parquet sequences are vectorized with bounded I/O concurrency and written to a resumable float16 memory map in recoverable chunks. Cache progress and every rejected sample remain inspectable.


In [ ]:
def append_jsonl(path: Path, row: Mapping[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(dict(row), ensure_ascii=False, default=str) + "\n")

def append_csv_row(path: Path, row: Mapping[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    flat = {key: value for key, value in row.items() if isinstance(value, (str, int, float, bool)) or value is None}
    write_header = not path.exists()
    with path.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(flat))
        if write_header: writer.writeheader()
        writer.writerow(flat)

def atomic_json_write(path: Path, payload: Mapping[str, Any]) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(dict(payload), indent=2, default=str), encoding="utf-8")
    os.replace(temporary, path)

def prepare_source_cache_item(path: Path, cfg: Config) -> Tuple[np.ndarray, int]:
    return preprocess_hands(
        source_parquet_hands(path), cfg.max_frames, cfg.min_frames,
        cfg.padding_value, cfg.max_input_frames,
    )

def build_source_cache(source_root: Path, cache_dir: Path, cfg: Config) -> Tuple[pd.DataFrame, Path, Path]:
    if cfg.cache_workers <= 0 or cfg.cache_chunk_size <= 0 or cfg.cache_flush_chunks <= 0:
        raise ValueError("cache_workers, cache_chunk_size, and cache_flush_chunks must be positive")
    cache_dir.mkdir(parents=True, exist_ok=True)
    metadata = pd.read_csv(source_root / "train.csv")
    required = {"path", "sign", "participant_id"}
    if not required.issubset(metadata.columns):
        raise ValueError(f"train.csv must contain {sorted(required)}")
    relative_paths = metadata["path"].astype(str).tolist()
    data_path, lengths_path = cache_dir / "features.f16", cache_dir / "lengths.i16"
    accepted_path, state_path = cache_dir / "accepted.csv", cache_dir / "state.json"
    rejected_path = cache_dir / "rejected.jsonl"
    shape = (len(metadata), cfg.max_frames, cfg.feature_dim)
    resumable = data_path.exists() and lengths_path.exists() and state_path.exists()
    mode = "r+" if resumable else "w+"
    features = np.memmap(data_path, dtype=np.float16, mode=mode, shape=shape)
    lengths = np.memmap(lengths_path, dtype=np.int16, mode=mode, shape=(len(metadata),))
    start = 0
    if mode == "w+":
        lengths[:] = 0
        features.flush(); lengths.flush()
        rejected_path.unlink(missing_ok=True)
    else:
        state = json.loads(state_path.read_text())
        if state.get("preprocessing") != PREPROCESSING_CONTRACT or state.get("rows") != len(metadata):
            raise RuntimeError("Source cache metadata does not match this dataset or preprocessing contract")
        start = int(state.get("next_index", 0))
        if not 0 <= start <= len(metadata):
            raise RuntimeError(f"Invalid source cache next_index: {start}")

    worker_count = max(1, min(cfg.cache_workers, os.cpu_count() or 1))
    progress = tqdm(total=len(metadata) - start, desc="Building temporal hand-feature cache")
    chunks_since_flush = 0
    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        for chunk_start in range(start, len(metadata), cfg.cache_chunk_size):
            chunk_end = min(len(metadata), chunk_start + cfg.cache_chunk_size)
            futures = {
                executor.submit(prepare_source_cache_item, source_root / relative_paths[index], cfg): index
                for index in range(chunk_start, chunk_end)
            }
            chunk_rejections = []
            for future in as_completed(futures):
                index = futures[future]
                try:
                    sample, valid_length = future.result()
                    features[index] = sample.astype(np.float16)
                    lengths[index] = valid_length
                except Exception as exc:
                    lengths[index] = 0
                    chunk_rejections.append(
                        {"index": index, "path": relative_paths[index], "reason": type(exc).__name__, "detail": str(exc)}
                    )
                progress.update(1)
            if chunk_rejections:
                with rejected_path.open("a", encoding="utf-8") as handle:
                    for rejection in sorted(chunk_rejections, key=lambda item: item["index"]):
                        handle.write(json.dumps(rejection, ensure_ascii=False, default=str) + "\n")
            chunks_since_flush += 1
            should_flush = chunks_since_flush >= cfg.cache_flush_chunks or chunk_end == len(metadata)
            if should_flush:
                features.flush(); lengths.flush()
                accepted_count = int(np.count_nonzero(lengths[:chunk_end]))
                atomic_json_write(state_path, {
                    "preprocessing": PREPROCESSING_CONTRACT,
                    "rows": len(metadata),
                    "next_index": chunk_end,
                    "accepted": accepted_count,
                    "rejected": chunk_end - accepted_count,
                    "complete": chunk_end == len(metadata),
                    "cache_workers": worker_count,
                    "cache_chunk_size": cfg.cache_chunk_size,
                })
                chunks_since_flush = 0
    progress.close()

    accepted_indices = np.flatnonzero(np.asarray(lengths) > 0)
    accepted_rows = metadata.iloc[accepted_indices][["sign", "participant_id", "path"]].copy()
    accepted_rows.insert(0, "cache_index", accepted_indices)
    temporary_accepted = accepted_path.with_suffix(accepted_path.suffix + ".tmp")
    accepted_rows.to_csv(temporary_accepted, index=False)
    os.replace(temporary_accepted, accepted_path)
    return accepted_rows.reset_index(drop=True), data_path, lengths_path

class MemmapDataset(Dataset):
    def __init__(self, rows: pd.DataFrame, data_path: Path, lengths_path: Path, total_rows: int, label_map: Mapping[str, int], cfg: Config):
        self.rows = rows.reset_index(drop=True)
        self.data = np.memmap(data_path, np.float16, "r", shape=(total_rows, cfg.max_frames, cfg.feature_dim))
        self.lengths = np.memmap(lengths_path, np.int16, "r", shape=(total_rows,))
        self.label_map, self.cfg = dict(label_map), cfg
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row = self.rows.iloc[index]
        cache_index = int(row.cache_index)
        return torch.from_numpy(np.array(self.data[cache_index], dtype=np.float32)), int(self.lengths[cache_index]), self.label_map[str(row.sign)]


## Target Dataset Audit 🛡️

Landmark records come from [ASL-preprocessing 7 output](https://www.kaggle.com/code/abdelrhmankaram/asl-preprocessing-7/output). Object arrays are deserialized with an explicit global allow-list. Invalid files are reported instead of silently entering training.


In [ ]:
class RestrictedNumpyUnpickler(pickle.Unpickler):
    ALLOWED = {
        ("numpy", "ndarray"), ("numpy", "dtype"),
        ("numpy.core.multiarray", "_reconstruct"), ("numpy._core.multiarray", "_reconstruct"),
        ("builtins", "set"), ("builtins", "slice"),
    }
    def find_class(self, module, name):
        if (module, name) not in self.ALLOWED:
            raise pickle.UnpicklingError(f"Blocked pickle global: {module}.{name}")
        return super().find_class(module, name)

def restricted_load_npy(path: Path) -> Any:
    if path.stat().st_size == 0:
        raise ValueError("Empty NPY file")
    with path.open("rb") as handle:
        version = np.lib.format.read_magic(handle)
        if version == (1, 0):
            shape, fortran, dtype = np.lib.format.read_array_header_1_0(handle)
        elif version in {(2, 0), (3, 0)}:
            shape, fortran, dtype = np.lib.format.read_array_header_2_0(handle)
        else:
            raise ValueError(f"Unsupported NPY version: {version}")
        if not dtype.hasobject:
            handle.seek(0)
            return np.load(handle, allow_pickle=False)
        value = RestrictedNumpyUnpickler(handle).load()
        if isinstance(value, np.ndarray) and value.shape != shape:
            raise ValueError(f"Object array header shape {shape} does not match payload {value.shape}")
        return value

def unpack_target_object(value: Any, cfg: Config) -> Tuple[str, np.ndarray]:
    if isinstance(value, np.ndarray) and value.dtype == object:
        value = value.tolist()
    if isinstance(value, dict):
        label = value.get("label", value.get("sign"))
        landmarks = value.get("landmark", value.get("landmarks"))
    elif isinstance(value, (list, tuple)) and len(value) >= 2:
        label, landmarks = value[0], value[1]
    else:
        raise ValueError("Expected an object containing label and landmarks")
    label = str(label).strip()
    landmarks = np.asarray(landmarks, dtype=np.float32)
    if not label: raise ValueError("Label is empty")
    if landmarks.ndim != 3 or landmarks.shape[1:] != (100, 3):
        raise ValueError(f"Expected T x 100 x 3 landmarks, received {landmarks.shape}")
    if not np.isfinite(landmarks).all(): raise ValueError("Landmarks contain non-finite values")
    rs, re = cfg.target_right_slice; ls, le = cfg.target_left_slice
    if re - rs != 21 or le - ls != 21: raise ValueError("Each configured target hand slice must contain 21 landmarks")
    hands = np.stack([landmarks[:, ls:le], landmarks[:, rs:re]], axis=1)  # canonical left, right
    return label, hands

def audit_target_dataset(root: Path, output_dir: Path, cfg: Config) -> Tuple[np.ndarray, np.ndarray, List[str], List[str]]:
    samples, lengths, labels, paths = [], [], [], []
    rejected = output_dir / "target_rejected.jsonl"
    for path in tqdm(sorted(root.rglob("*.npy")), desc="Auditing target NPY"):
        try:
            label, hands = unpack_target_object(restricted_load_npy(path), cfg)
            sample, length = preprocess_hands(hands, cfg.max_frames, cfg.min_frames, cfg.padding_value, cfg.max_input_frames)
            samples.append(sample); lengths.append(length); labels.append(label); paths.append(str(path))
        except Exception as exc:
            append_jsonl(rejected, {"path": str(path), "reason": type(exc).__name__, "detail": str(exc)})
    if not samples: raise RuntimeError(f"No valid target samples were found under {root}")
    return np.stack(samples), np.asarray(lengths, np.int64), labels, paths

class ArrayDataset(Dataset):
    def __init__(self, samples, lengths, labels, indices, label_map):
        self.samples, self.lengths, self.labels = samples, lengths, labels
        self.indices, self.label_map = np.asarray(indices), dict(label_map)
    def __len__(self): return len(self.indices)
    def __getitem__(self, index):
        i = int(self.indices[index])
        return torch.from_numpy(np.asarray(self.samples[i], np.float32)), int(self.lengths[i]), self.label_map[self.labels[i]]


## Model and Objectives 🧠

+The encoder combines a shared per-hand temporal branch with masked Transformer fusion. Classification is used only as an auxiliary training signal.


In [ ]:
def split_labels(labels: Sequence[str], seed: int, ratios=(0.8, 0.1, 0.1)) -> Dict[str, List[str]]:
    unique = sorted(set(map(str, labels)))
    if len(unique) < 3: raise ValueError("At least three distinct labels are required for class-disjoint splits")
    rng = random.Random(seed); rng.shuffle(unique)
    n_val = max(1, round(len(unique) * ratios[1])); n_test = max(1, round(len(unique) * ratios[2]))
    n_train = len(unique) - n_val - n_test
    if n_train < 1: n_train, n_val = 1, len(unique) - n_test - 1
    result = {"train": unique[:n_train], "validation": unique[n_train:n_train+n_val], "test": unique[n_train+n_val:]}
    if set(result["train"]) & set(result["validation"]) or set(result["train"]) & set(result["test"]) or set(result["validation"]) & set(result["test"]):
        raise AssertionError("Class leakage detected")
    return result

class PKBatchSampler(Sampler[List[int]]):
    def __init__(self, labels: Sequence[int], p: int, k: int, batches: int, seed: int):
        self.p, self.k, self.batches, self.seed, self.epoch = p, k, batches, seed, 0
        self.groups: Dict[int, List[int]] = {}
        for index, label in enumerate(labels): self.groups.setdefault(int(label), []).append(index)
        if len(self.groups) < p: raise ValueError(f"PK sampling needs {p} classes, found {len(self.groups)}")
    def set_epoch(self, epoch): self.epoch = int(epoch)
    def __len__(self): return self.batches
    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)
        classes = list(self.groups)
        for _ in range(self.batches):
            batch = []
            for label in rng.sample(classes, self.p):
                members = self.groups[label]
                batch.extend(rng.sample(members, self.k) if len(members) >= self.k else rng.choices(members, k=self.k))
            yield batch

def pk_batches(labels, p, k, batches, seed):
    return PKBatchSampler(labels, p, k, batches, seed)

class TemporalBlock(nn.Module):
    def __init__(self, channels: int, dilation: int):
        super().__init__()
        self.depthwise = nn.Conv1d(channels, channels, 3, padding=dilation, dilation=dilation, groups=channels)
        self.pointwise = nn.Conv1d(channels, channels, 1)
        self.norm = nn.BatchNorm1d(channels)
    def forward(self, x): return F.gelu(self.norm(self.pointwise(self.depthwise(x)))) + x

class HandEncoder(nn.Module):
    def __init__(self, feature_dim=168, hand_dim=96, fusion_dim=192, embedding_dim=128, max_frames=64):
        super().__init__()
        if feature_dim != 168: raise ValueError("hand168-temporal requires feature_dim=168")
        self.model_config = dict(feature_dim=feature_dim, hand_dim=hand_dim, fusion_dim=fusion_dim, embedding_dim=embedding_dim, max_frames=max_frames)
        self.input_projection = nn.Linear(84, hand_dim)
        self.tcn = nn.Sequential(TemporalBlock(hand_dim, 1), TemporalBlock(hand_dim, 2), TemporalBlock(hand_dim, 4))
        self.fusion = nn.Linear(hand_dim * 2, fusion_dim)
        self.position = nn.Parameter(torch.zeros(1, max_frames, fusion_dim))
        layer = nn.TransformerEncoderLayer(fusion_dim, 4, fusion_dim * 4, 0.1, batch_first=True, norm_first=True, activation="gelu")
        self.transformer = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.output = nn.Linear(fusion_dim, embedding_dim)
        nn.init.trunc_normal_(self.position, std=0.02)
    def forward(self, x, lengths):
        if x.ndim != 3 or x.shape[-1] != 168: raise ValueError(f"Expected B x T x 168, received {tuple(x.shape)}")
        steps = x.shape[1]; mask = torch.arange(steps, device=x.device)[None] >= lengths[:, None]
        x = x.masked_fill(mask[..., None], 0.0).reshape(x.shape[0], steps, 2, 84)
        streams = []
        for hand in range(2):
            stream = self.input_projection(x[:, :, hand]).transpose(1, 2)
            streams.append(self.tcn(stream).transpose(1, 2))
        fused = self.fusion(torch.cat(streams, dim=-1)) + self.position[:, :steps]
        fused = self.transformer(fused, src_key_padding_mask=mask)
        output = F.normalize(self.output(fused), dim=-1)
        return output.masked_fill(mask[..., None], 0.0)

class TrainingModel(nn.Module):
    def __init__(self, encoder: HandEncoder, classes: int):
        super().__init__(); self.encoder, self.classifier = encoder, nn.Linear(encoder.model_config["embedding_dim"], classes)
    def forward(self, x, lengths):
        frames = self.encoder(x, lengths)
        mask = torch.arange(frames.shape[1], device=frames.device)[None] < lengths[:, None]
        pooled = (frames * mask[..., None]).sum(1) / lengths.clamp_min(1)[:, None]
        pooled = F.normalize(pooled, dim=-1)
        return frames, pooled, self.classifier(pooled)

def supervised_contrastive_loss(features, labels, temperature=0.1):
    features = F.normalize(features, dim=-1); logits = features @ features.T / temperature
    eye = torch.eye(len(labels), dtype=torch.bool, device=labels.device)
    positives = labels[:, None].eq(labels[None]) & ~eye
    logits = logits.masked_fill(eye, -torch.inf)
    log_prob = logits - torch.logsumexp(logits, dim=1, keepdim=True)
    valid = positives.any(1)
    return -(log_prob.masked_fill(~positives, 0).sum(1)[valid] / positives.sum(1)[valid]).mean() if valid.any() else features.sum() * 0

def batch_hard_triplet_loss(features, labels, margin=0.2):
    distance = 1.0 - F.normalize(features, dim=-1) @ F.normalize(features, dim=-1).T
    eye = torch.eye(len(labels), dtype=torch.bool, device=labels.device)
    pos = labels[:, None].eq(labels[None]) & ~eye; neg = ~labels[:, None].eq(labels[None])
    valid = pos.any(1) & neg.any(1)
    hardest_pos = distance.masked_fill(~pos, -torch.inf).max(1).values
    hardest_neg = distance.masked_fill(~neg, torch.inf).min(1).values
    return F.relu(hardest_pos[valid] - hardest_neg[valid] + margin).mean() if valid.any() else features.sum() * 0

def augment_features(x, lengths):
    x = x.clone(); batch = x.shape[0]
    for i in range(batch):
        length = int(lengths[i]); view = x[i, :length].reshape(length, 2, 21, 4)
        if torch.rand(()) < 0.5: view[..., 0].mul_(-1)
        if torch.rand(()) < 0.5:
            view[..., :3].mul_(torch.empty((), device=x.device).uniform_(0.95, 1.05))
        if length > 4 and torch.rand(()) < 0.3:
            start = int(torch.randint(0, length - 1, ()).item()); view[start:min(length, start + max(1, length // 10))].zero_()
        if torch.rand(()) < 0.3:
            hand = int(torch.randint(0, 2, ()).item()); landmark = int(torch.randint(1, 21, ()).item()); view[:, hand, landmark].zero_()
    return x


## Representation Training 🏋️

+Classes are split before sampling. Unseen-class validation selects the checkpoint using prototype-style retrieval, not the temporary classifier.


In [ ]:
@torch.inference_mode()
def encode_loader(model, loader, device):
    model.eval(); vectors, labels, sequences, lengths_out = [], [], [], []
    for x, lengths, y in loader:
        x, lengths = x.to(device), lengths.to(device)
        frames, pooled, _ = model(x, lengths)
        vectors.append(pooled.cpu().numpy()); labels.extend(y.numpy().tolist())
        sequences.extend([frames[i, :int(lengths[i])].cpu().numpy() for i in range(len(x))]); lengths_out.extend(lengths.cpu().tolist())
    return np.concatenate(vectors), np.asarray(labels), sequences, np.asarray(lengths_out)

def cosine_distance(a, b): return 1.0 - np.clip(a @ b.T, -1.0, 1.0)

def make_reference_query(labels, seed, references=1, max_queries=4, groups=None):
    rng = random.Random(seed); ref, query = [], []
    for label in sorted(set(labels.tolist())):
        indices = np.flatnonzero(labels == label).tolist(); rng.shuffle(indices)
        selected = indices[:references]
        if groups is None:
            candidates = indices[references:]
        else:
            reference_groups = {str(groups[i]) for i in selected}
            candidates = [i for i in indices if i not in selected and str(groups[i]) not in reference_groups]
        if selected and candidates:
            ref.extend(selected); query.extend(candidates[:max_queries])
    return np.asarray(ref), np.asarray(query)

def pooled_retrieval(vectors, labels, seed=0, references=1, max_queries=4, groups=None):
    ref, query = make_reference_query(labels, seed, references, max_queries, groups)
    if not len(query): return {"recall_at_1": 0.0, "queries": 0, "method": "pooled_cosine"}
    distances = cosine_distance(vectors[query], vectors[ref])
    predictions = labels[ref][distances.argmin(1)]
    return {"recall_at_1": float(np.mean(predictions == labels[query])), "queries": int(len(query)), "method": "pooled_cosine"}

def constrained_dtw(a: np.ndarray, b: np.ndarray, window: int = 12) -> float:
    n, m = len(a), len(b); window = max(window, abs(n - m))
    costs = np.full((n + 1, m + 1), np.inf, np.float64); steps = np.zeros((n + 1, m + 1), np.int32); costs[0, 0] = 0
    for i in range(1, n + 1):
        for j in range(max(1, i - window), min(m, i + window) + 1):
            options = [(costs[i-1, j], steps[i-1, j]), (costs[i, j-1], steps[i, j-1]), (costs[i-1, j-1], steps[i-1, j-1])]
            best_cost, best_steps = min(options, key=lambda item: item[0])
            costs[i, j] = best_cost + float(np.linalg.norm(a[i-1] - b[j-1])); steps[i, j] = best_steps + 1
    return float(costs[n, m] / max(1, steps[n, m]))

def topk_dtw_retrieval(vectors, sequences, labels, seed, top_k=20, window=12, max_queries=2, groups=None):
    ref, query = make_reference_query(labels, seed, 1, max_queries, groups)
    correct = 0
    for qi in query:
        pooled = cosine_distance(vectors[qi:qi+1], vectors[ref])[0]
        shortlist = ref[np.argsort(pooled)[:min(top_k, len(ref))]]
        prediction = labels[min(shortlist, key=lambda ri: constrained_dtw(sequences[qi], sequences[ri], window))]
        correct += int(prediction == labels[qi])
    return {"recall_at_1": correct / len(query) if len(query) else 0.0, "queries": int(len(query)), "method": "topk_dtw"}

def atomic_torch_save(payload, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True); temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary); os.replace(temporary, path)

def split_identity(split):
    return hashlib.sha256(json.dumps(split, sort_keys=True).encode()).hexdigest()

def train_recipe(name, model, train_dataset, train_labels, validation_loader, settings, split, output_dir, device, cfg, validation_groups=None):
    output_dir.mkdir(parents=True, exist_ok=True)
    sampler = PKBatchSampler(train_labels, settings.classes_per_batch, settings.samples_per_class, settings.batches_per_epoch, cfg.seed)
    loader = DataLoader(train_dataset, batch_sampler=sampler, num_workers=cfg.num_workers, pin_memory=True)
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings.learning_rate, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, settings.epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)
    latest = output_dir / "latest.pt"; start_epoch, best = 0, -1.0
    expected_split = split_identity(split)
    if cfg.resume and latest.exists():
        state = torch.load(latest, map_location="cpu", weights_only=False)
        if state["schema"] != CHECKPOINT_SCHEMA or state["preprocessing"] != PREPROCESSING_CONTRACT or state["split_identity"] != expected_split:
            raise RuntimeError(f"Refusing incompatible resume checkpoint: {latest}")
        model.load_state_dict(state["model"]); optimizer.load_state_dict(state["optimizer"]); scheduler.load_state_dict(state["scheduler"]); scaler.load_state_dict(state["scaler"])
        start_epoch, best = state["epoch"] + 1, state["best_recall"]
    if cfg.use_data_parallel:
        if torch.cuda.device_count() < 2: raise RuntimeError("use_data_parallel=True requires at least two visible CUDA devices")
        model = nn.DataParallel(model)
    core_model = model.module if isinstance(model, nn.DataParallel) else model
    for epoch in range(start_epoch, settings.epochs):
        sampler.set_epoch(epoch); model.train()
        frozen = epoch < settings.freeze_epochs
        for parameter in list(core_model.encoder.input_projection.parameters()) + list(core_model.encoder.tcn[0].parameters()): parameter.requires_grad_(not frozen)
        running = 0.0
        progress = tqdm(loader, desc=f"{name} {epoch+1}/{settings.epochs}", leave=False)
        for batch_index, (x, lengths, labels) in enumerate(progress):
            x, lengths, labels = x.to(device, non_blocking=True), lengths.to(device), labels.to(device)
            x = augment_features(x, lengths); optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=cfg.use_amp):
                _, pooled, logits = model(x, lengths)
                ce = F.cross_entropy(logits, labels); supcon = supervised_contrastive_loss(pooled, labels); triplet = batch_hard_triplet_loss(pooled, labels)
                loss = ce + 0.25 * supcon + 0.5 * triplet
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); running += loss.detach().item()
            progress.set_postfix(loss=f"{running/(batch_index+1):.4f}")
            if (batch_index + 1) % 25 == 0:
                (output_dir / "status.json").write_text(json.dumps({"epoch": epoch, "last_batch": batch_index, "cuda_bytes": torch.cuda.memory_allocated()}, indent=2))
        scheduler.step()
        vectors, labels_np, sequences, _ = encode_loader(model, validation_loader, device)
        use_dtw = settings.dtw_every and (epoch + 1) % settings.dtw_every == 0
        metrics = topk_dtw_retrieval(vectors, sequences, labels_np, cfg.seed, cfg.top_k, cfg.dtw_window, groups=validation_groups) if use_dtw else pooled_retrieval(vectors, labels_np, cfg.seed, cfg.source_reference_per_class, cfg.source_query_per_class, validation_groups)
        recall = metrics["recall_at_1"]; best = max(best, recall)
        payload = {"schema": CHECKPOINT_SCHEMA, "preprocessing": PREPROCESSING_CONTRACT, "model_config": core_model.encoder.model_config, "model": core_model.state_dict(), "encoder": core_model.encoder.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "epoch": epoch, "best_recall": best, "metrics": metrics, "split_identity": expected_split, "seed": cfg.seed}
        epoch_path = output_dir / f"epoch_{epoch+1:03d}.pt"; atomic_torch_save(payload, epoch_path); atomic_torch_save(payload, latest)
        if recall >= best: atomic_torch_save(payload, output_dir / "best.pt")
        metric_row = {"epoch": epoch + 1, "loss": running / len(loader), **metrics}
        append_jsonl(output_dir / "metrics.jsonl", metric_row); append_csv_row(output_dir / "metrics.csv", metric_row)
        log_line = f"{name}: epoch={epoch+1} loss={running/len(loader):.4f} recall@1={recall:.4f}"
        with (output_dir / "train.log").open("a", encoding="utf-8") as handle: handle.write(log_line + "\n")
        print(log_line)
        gc.collect(); torch.cuda.empty_cache()
    if not (output_dir / "best.pt").exists(): raise RuntimeError(f"No best checkpoint was produced for {name}")
    return output_dir / "best.pt"


In [ ]:
# Prepare the source corpus and deterministic class-disjoint splits.
SOURCE_OUT = WORK_DIR / "representation_training"
source_rows, source_data_path, source_lengths_path = build_source_cache(SOURCE_ROOT, WORK_DIR / "source_cache", CFG)
source_split = split_labels(source_rows.sign.tolist(), CFG.seed)
(SOURCE_OUT / "splits.json").parent.mkdir(parents=True, exist_ok=True)
(SOURCE_OUT / "splits.json").write_text(json.dumps(source_split, indent=2))

source_train = source_rows[source_rows.sign.isin(source_split["train"])].reset_index(drop=True)
source_val = source_rows[source_rows.sign.isin(source_split["validation"])].reset_index(drop=True)
source_train_map = {label: i for i, label in enumerate(source_split["train"])}
source_val_map = {label: i for i, label in enumerate(source_split["validation"])}
total_source_rows = len(pd.read_csv(SOURCE_ROOT / "train.csv"))
source_train_ds = MemmapDataset(source_train, source_data_path, source_lengths_path, total_source_rows, source_train_map, CFG)
source_val_ds = MemmapDataset(source_val, source_data_path, source_lengths_path, total_source_rows, source_val_map, CFG)
source_val_loader = DataLoader(source_val_ds, batch_size=128, shuffle=False, num_workers=0)
source_train_labels = [source_train_map[str(label)] for label in source_train.sign]

encoder = HandEncoder(embedding_dim=CFG.embedding_dim, max_frames=CFG.max_frames)
source_model = TrainingModel(encoder, len(source_train_map))
source_best = train_recipe(
    "representation", source_model, source_train_ds, source_train_labels, source_val_loader,
    CFG.source, source_split, SOURCE_OUT, DEVICE, CFG, source_val.participant_id.astype(str).tolist()
)

# Evaluate the selected source encoder once with participant-disjoint references and queries.
source_test = source_rows[source_rows.sign.isin(source_split["test"])].reset_index(drop=True)
source_test_map = {label: i for i, label in enumerate(source_split["test"])}
source_test_ds = MemmapDataset(source_test, source_data_path, source_lengths_path, total_source_rows, source_test_map, CFG)
source_test_loader = DataLoader(source_test_ds, batch_size=128, shuffle=False, num_workers=0)
source_eval_state = torch.load(source_best, map_location="cpu", weights_only=False)
source_model.load_state_dict(source_eval_state["model"])
source_vectors, source_test_labels, source_sequences, _ = encode_loader(source_model.to(DEVICE), source_test_loader, DEVICE)
source_test_metrics = topk_dtw_retrieval(
    source_vectors, source_sequences, source_test_labels, CFG.seed + 1, CFG.top_k, CFG.dtw_window,
    CFG.final_query_per_class, source_test.participant_id.astype(str).tolist()
)
(SOURCE_OUT / "test_metrics.json").write_text(json.dumps(source_test_metrics, indent=2))
print("Source test:", json.dumps(source_test_metrics))


## Domain Adaptation 🔧

+Only the best encoder is transferred. A new temporary classifier is created for target training labels; early encoder layers are briefly frozen, then fine-tuned.


In [ ]:
TARGET_OUT = WORK_DIR / "domain_adaptation"
target_samples, target_lengths, target_labels, target_paths = audit_target_dataset(TARGET_LANDMARKS, TARGET_OUT, CFG)
target_split = split_labels(target_labels, CFG.seed + 1)
TARGET_OUT.mkdir(parents=True, exist_ok=True)
(TARGET_OUT / "splits.json").write_text(json.dumps(target_split, indent=2))

indices_by_split = {name: [i for i, label in enumerate(target_labels) if label in labels] for name, labels in target_split.items()}
target_train_map = {label: i for i, label in enumerate(target_split["train"])}
target_val_map = {label: i for i, label in enumerate(target_split["validation"])}
target_train_ds = ArrayDataset(target_samples, target_lengths, target_labels, indices_by_split["train"], target_train_map)
target_val_ds = ArrayDataset(target_samples, target_lengths, target_labels, indices_by_split["validation"], target_val_map)
target_val_loader = DataLoader(target_val_ds, batch_size=128, shuffle=False, num_workers=0)
target_train_numeric = [target_train_map[target_labels[i]] for i in indices_by_split["train"]]

source_state = torch.load(source_best, map_location="cpu", weights_only=False)
if source_state["schema"] != CHECKPOINT_SCHEMA or source_state["preprocessing"] != PREPROCESSING_CONTRACT:
    raise RuntimeError("The source encoder checkpoint is incompatible with this notebook")
adapted_encoder = HandEncoder(**source_state["model_config"])
adapted_encoder.load_state_dict(source_state["encoder"])
adaptation_model = TrainingModel(adapted_encoder, len(target_train_map))
adaptation_best = train_recipe("adaptation", adaptation_model, target_train_ds, target_train_numeric, target_val_loader, CFG.adaptation, target_split, TARGET_OUT, DEVICE, CFG)


## Retrieval Evaluation 📊

Final evaluation first shortlists prototypes with pooled cosine distance and then reranks them with constrained frame-level DTW.


In [ ]:
final_state = torch.load(adaptation_best, map_location="cpu", weights_only=False)
final_encoder = HandEncoder(**final_state["model_config"])
final_encoder.load_state_dict(final_state["encoder"])
test_map = {label: i for i, label in enumerate(target_split["test"])}
target_test_ds = ArrayDataset(target_samples, target_lengths, target_labels, indices_by_split["test"], test_map)
target_test_loader = DataLoader(target_test_ds, batch_size=128, shuffle=False, num_workers=0)
evaluation_model = TrainingModel(final_encoder, max(1, len(test_map))).to(DEVICE)
test_vectors, test_numeric, test_sequences, _ = encode_loader(evaluation_model, target_test_loader, DEVICE)
final_metrics = topk_dtw_retrieval(test_vectors, test_sequences, test_numeric, CFG.seed + 2, CFG.top_k, CFG.dtw_window, CFG.final_query_per_class)
(WORK_DIR / "final_evaluation.json").write_text(json.dumps(final_metrics, indent=2))
print(json.dumps(final_metrics, indent=2))


## Training Curves 📈

Training metrics are rendered from the persisted CSV files. Figures are shown inline and saved as reproducible PNG artifacts.


In [ ]:
def generate_training_charts(
    work_dir: Path,
    source_test: Optional[Mapping[str, Any]] = None,
    target_test: Optional[Mapping[str, Any]] = None,
) -> List[Path]:
    import matplotlib.pyplot as plt

    figures_dir = work_dir / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)
    runs = [
        ("Representation Training", work_dir / "representation_training" / "metrics.csv", "#2563eb"),
        ("Domain Adaptation", work_dir / "domain_adaptation" / "metrics.csv", "#ea580c"),
    ]
    available = []
    for label, path, color in runs:
        if not path.exists():
            print(f"Chart input is unavailable: {path}")
            continue
        frame = pd.read_csv(path)
        required = {"epoch", "loss", "recall_at_1"}
        if frame.empty or not required.issubset(frame.columns):
            print(f"Chart input has no usable metrics: {path}")
            continue
        frame = frame.copy()
        frame["training_run"] = label
        available.append((label, frame, color))
    if not available:
        print("No training metrics are available; chart generation was skipped")
        return []

    plot_data = pd.concat([frame for _, frame, _ in available], ignore_index=True)
    plot_data.to_csv(figures_dir / "plot_data.csv", index=False)
    outputs = [figures_dir / "plot_data.csv"]

    style_name = "seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default"
    with plt.style.context(style_name):
        figure, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
        for row, (expected_label, _, default_color) in enumerate(runs):
            match = next(((label, frame, color) for label, frame, color in available if label == expected_label), None)
            if match is None:
                axes[row, 0].set_visible(False); axes[row, 1].set_visible(False)
                continue
            label, frame, color = match
            axes[row, 0].plot(frame["epoch"], frame["loss"], color=color, linewidth=2)
            axes[row, 0].set(title=f"{label} — Loss", xlabel="Epoch", ylabel="Total loss")
            axes[row, 1].plot(frame["epoch"], frame["recall_at_1"], color=color, linewidth=2, marker="o", markersize=3)
            axes[row, 1].set(title=f"{label} — Validation Recall@1", xlabel="Epoch", ylabel="Recall@1", ylim=(0, 1.02))
        curves_path = figures_dir / "training_curves.png"
        figure.suptitle("Training Progress", fontsize=15, fontweight="bold")
        figure.savefig(curves_path, dpi=180, bbox_inches="tight")
        plt.show(); plt.close(figure); outputs.append(curves_path)

        evaluation = []
        for label, metrics, color in (
            ("Source Retrieval", source_test, "#2563eb"),
            ("Target Retrieval", target_test, "#ea580c"),
        ):
            if metrics is not None and "recall_at_1" in metrics:
                evaluation.append((label, float(metrics["recall_at_1"]), color))
        if evaluation:
            figure, axis = plt.subplots(figsize=(8, 4.8), constrained_layout=True)
            bars = axis.bar([item[0] for item in evaluation], [item[1] for item in evaluation], color=[item[2] for item in evaluation], width=0.58)
            axis.set(title="Final Retrieval Evaluation", ylabel="Recall@1", ylim=(0, 1.08))
            axis.bar_label(bars, labels=[f"{item[1]:.3f}" for item in evaluation], padding=4)
            summary_path = figures_dir / "retrieval_summary.png"
            figure.savefig(summary_path, dpi=180, bbox_inches="tight")
            plt.show(); plt.close(figure); outputs.append(summary_path)
        else:
            print("Final retrieval metrics are unavailable; the summary chart was skipped")
    return outputs

try:
    chart_outputs = generate_training_charts(WORK_DIR, source_test_metrics, final_metrics)
    for chart_path in chart_outputs: print(f"Chart artifact: {chart_path}")
except Exception as exc:
    chart_outputs = []
    print(f"Chart generation was skipped without interrupting export: {type(exc).__name__}: {exc}")


## Export 📦

The standalone encoder includes its architecture, preprocessing contract, and SHA-256 fingerprint. Dynamic prototypes must match both the fingerprint and preprocessing contract.


In [ ]:
def encoder_fingerprint(state_dict: Mapping[str, torch.Tensor]) -> str:
    digest = hashlib.sha256()
    for name in sorted(state_dict):
        tensor = state_dict[name].detach().cpu().contiguous()
        digest.update(name.encode()); digest.update(str(tensor.dtype).encode()); digest.update(np.asarray(tensor.shape, np.int64).tobytes()); digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()

encoder_state = final_state["encoder"]
fingerprint = encoder_fingerprint(encoder_state)
export = {
    "schema": CHECKPOINT_SCHEMA,
    "preprocessing": PREPROCESSING_CONTRACT,
    "model_config": final_state["model_config"],
    "encoder": encoder_state,
    "encoder_fingerprint": fingerprint,
    "input_contract": {"dtype": "float32", "shape": ["B", 64, 168], "padding_value": CFG.padding_value},
    "output_contract": {"dtype": "float32", "shape": ["B", 64, 128], "padding_output": 0.0},
}
atomic_torch_save(export, WORK_DIR / "signlang_det_encoder.pt")
(WORK_DIR / "run_config.json").write_text(json.dumps(asdict(CFG), indent=2, default=str))
(WORK_DIR / "environment.json").write_text(json.dumps({"python": sys.version, "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name()}, indent=2))
print(f"Encoder exported to {WORK_DIR / 'signlang_det_encoder.pt'}\nSHA-256: {fingerprint}")


## Checkpoint Cleanup 🧹

Cleanup runs only after the exported encoder has been reloaded and verified. Best checkpoints and all reproducibility artifacts are retained.


In [ ]:
def verify_exported_encoder(path: Path) -> Dict[str, Any]:
    payload = torch.load(path, map_location="cpu", weights_only=False)
    required = {"schema", "preprocessing", "model_config", "encoder", "encoder_fingerprint"}
    missing = sorted(required - payload.keys())
    if missing: raise RuntimeError(f"Exported encoder is missing fields: {missing}")
    if payload["schema"] != CHECKPOINT_SCHEMA or payload["preprocessing"] != PREPROCESSING_CONTRACT:
        raise RuntimeError("Exported encoder schema or preprocessing contract is incompatible")
    actual_fingerprint = encoder_fingerprint(payload["encoder"])
    if actual_fingerprint != payload["encoder_fingerprint"]:
        raise RuntimeError("Exported encoder fingerprint verification failed")
    model = HandEncoder(**payload["model_config"])
    model.load_state_dict(payload["encoder"], strict=True); model.eval()
    sample = torch.full((1, CFG.max_frames, CFG.feature_dim), CFG.padding_value, dtype=torch.float32)
    sample[:, :CFG.min_frames] = 0.0
    with torch.inference_mode(): output = model(sample, torch.tensor([CFG.min_frames]))
    expected_shape = (1, CFG.max_frames, CFG.embedding_dim)
    if tuple(output.shape) != expected_shape or not torch.isfinite(output).all():
        raise RuntimeError(f"Exported encoder smoke test failed: output shape={tuple(output.shape)}")
    if torch.count_nonzero(output[:, CFG.min_frames:]).item() != 0:
        raise RuntimeError("Exported encoder produced non-zero padding features")
    return {
        "path": str(path), "fingerprint": actual_fingerprint,
        "output_shape": list(output.shape), "verified": True,
    }

def cleanup_checkpoints(work_dir: Path, run_dirs: Sequence[Path], keep_latest: bool = False) -> Dict[str, Any]:
    run_dirs = [Path(path) for path in run_dirs]
    missing_best = [str(path / "best.pt") for path in run_dirs if not (path / "best.pt").is_file()]
    if missing_best: raise RuntimeError(f"Checkpoint cleanup refused; missing retained best checkpoints: {missing_best}")
    targets = []
    for run_dir in run_dirs:
        targets.extend(sorted(run_dir.glob("epoch_*.pt")))
        latest = run_dir / "latest.pt"
        if not keep_latest and latest.is_file(): targets.append(latest)
    deleted, released_bytes = [], 0
    for path in targets:
        size = path.stat().st_size
        path.unlink()
        released_bytes += size
        try: display_path = str(path.relative_to(work_dir))
        except ValueError: display_path = str(path)
        deleted.append({"path": display_path, "bytes": size})
    retained = []
    for run_dir in run_dirs:
        retained.append(str(run_dir / "best.pt"))
        latest = run_dir / "latest.pt"
        if latest.is_file(): retained.append(str(latest))
    report = {
        "deleted_files": len(deleted), "released_bytes": released_bytes,
        "released_gib": released_bytes / (1024 ** 3), "deleted": deleted,
        "retained": retained, "keep_latest": keep_latest,
    }
    atomic_json_write(work_dir / "checkpoint_cleanup.json", report)
    return report

export_verification = verify_exported_encoder(WORK_DIR / "signlang_det_encoder.pt")
if CFG.cleanup_epoch_checkpoints:
    cleanup_report = cleanup_checkpoints(
        WORK_DIR, [SOURCE_OUT, TARGET_OUT], keep_latest=CFG.keep_latest_checkpoint,
    )
    print(
        f"Checkpoint cleanup removed {cleanup_report['deleted_files']} files and released "
        f"{cleanup_report['released_gib']:.3f} GiB. Best checkpoints were retained."
    )
else:
    cleanup_report = {"skipped": True, "reason": "cleanup_epoch_checkpoints=False"}
    atomic_json_write(WORK_DIR / "checkpoint_cleanup.json", cleanup_report)
    print("Checkpoint cleanup was skipped by configuration")


## Final Output Assembly 🚚

A deterministic subset of target-training inputs is packaged for RKNN INT8 calibration. The working directory is then reduced to the public output allowlist; any missing required artifact stops the run.


In [ ]:
import tarfile, tempfile

FINAL_OUTPUTS = {
    "signlang_det_encoder.pt",
    "int8_calibration.tar.gz",
    "figures/training_curves.png",
    "figures/retrieval_summary.png",
    "representation_training/metrics.csv",
    "domain_adaptation/metrics.csv",
    "representation_training/train.log",
    "domain_adaptation/train.log",
}

def create_int8_calibration_archive(dataset: Dataset, output_path: Path, sample_count: int, seed: int) -> Path:
    if sample_count <= 0:
        raise ValueError("CFG.int8_calibration_samples must be positive")
    count = min(sample_count, len(dataset))
    if count == 0:
        raise RuntimeError("Target training dataset is empty; INT8 calibration cannot be prepared")
    selected = sorted(random.Random(seed).sample(range(len(dataset)), count))
    with tempfile.TemporaryDirectory() as directory:
        root = Path(directory); samples_dir = root / "samples"; samples_dir.mkdir()
        dataset_lines, manifest_rows = [], []
        for output_index, dataset_index in enumerate(selected):
            features, length, _ = dataset[dataset_index]
            feature_name = f"samples/features_{output_index:04d}.npy"
            length_name = f"samples/lengths_{output_index:04d}.npy"
            np.save(root / feature_name, features.numpy().astype(np.float32, copy=False)[None], allow_pickle=False)
            np.save(root / length_name, np.asarray([length], dtype=np.int32), allow_pickle=False)
            dataset_lines.append(f"{feature_name} {length_name}")
            manifest_rows.append({"calibration_index": output_index, "training_dataset_index": dataset_index, "valid_length": length})
        (root / "dataset.txt").write_text("\n".join(dataset_lines) + "\n", encoding="utf-8")
        with (root / "selection.csv").open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(manifest_rows[0])); writer.writeheader(); writer.writerows(manifest_rows)
        with tarfile.open(output_path, "w:gz") as bundle:
            for path in sorted(root.rglob("*")):
                if path.is_file(): bundle.add(path, arcname=str(path.relative_to(root)))
    return output_path

def prune_notebook_outputs(work_dir: Path, allowed: Sequence[str]) -> None:
    expected = {Path(item) for item in allowed}
    missing = sorted(str(path) for path in expected if not (work_dir / path).is_file())
    if missing:
        raise RuntimeError(f"Required final notebook outputs are missing: {missing}")
    for path in sorted(work_dir.rglob("*"), key=lambda item: len(item.parts), reverse=True):
        if path.is_file() and path.relative_to(work_dir) not in expected:
            path.unlink()
        elif path.is_dir():
            try: path.rmdir()
            except OSError: pass
    actual = {path.relative_to(work_dir) for path in work_dir.rglob("*") if path.is_file()}
    if actual != expected:
        raise RuntimeError(f"Final output allowlist mismatch: expected={sorted(map(str, expected))}, actual={sorted(map(str, actual))}")

create_int8_calibration_archive(
    target_train_ds, WORK_DIR / "int8_calibration.tar.gz", CFG.int8_calibration_samples, CFG.seed + 3
)
prune_notebook_outputs(WORK_DIR, FINAL_OUTPUTS)
print("Final notebook outputs:\n" + "\n".join(f"  - {path}" for path in sorted(FINAL_OUTPUTS)))


## Usage Notes 📖

- Attach the Google Isolated Sign Language Recognition dataset and the target dataset containing a `landmarks` directory before starting the session.
- Edit `CFG.source_root` and `CFG.target_landmarks` when the data is mounted outside the standard Kaggle paths configured by the uploader.
- Keep `CFG.num_workers = 0` for reliable Kaggle notebook restarts. A single T4 is the default; multi-GPU execution is intentionally opt-in.
- Interrupted runs resume compatible `latest.pt` checkpoints and continue an incomplete source cache.
- After successful export verification, the notebook packages up to `int8_calibration_samples` target-training inputs and removes every file outside the documented final output allowlist.
- Keep `signlang_det_encoder.pt`, `int8_calibration.tar.gz`, both charts, both metric CSV files, and both training logs together for delivery.
- Unknown-sign distance and margin thresholds are **not** derived from training loss. Calibrate them with separate known-query and unknown-action data before deployment.
- Each dynamic prototype should retain its label, valid length, frame features, pooled feature, preprocessing contract, and encoder fingerprint.
